In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from model1 import run_torch_version
import json

def result_record(alg_name, ret, dataset, param=''):
    # 1. 动态构造键名 (Key)
    key = f"{dataset}_{alg_name}_{param}" if param else f"{dataset}_{alg_name}"
    
    # 2. 构造字典对象
    record_dict = {key: ret}
    
    # 3. 追加写入文件
    with open("result.jsonl", 'a') as f:
        # json.dumps 会自动给键加上双引号，并将元组 (0.6..., ...) 转换为列表 [0.6..., ...]
        f.write(json.dumps(record_dict) + '\n')

def modified_kmeans_fast_log_partitioned(mi_matrix, node_communities, tolerance=1e-7):
    # 1. 预处理：取对数拉伸低值区差异
    epsilon = 1e-9
    log_mi_matrix = np.log(np.clip(mi_matrix, epsilon, None))

    n = mi_matrix.shape[0]
    triu_indices = np.triu_indices(n, k=1)
    rows_all, cols_all = triu_indices
    all_log_values = log_mi_matrix[triu_indices]
    all_raw_values = mi_matrix[triu_indices]

    # 2. 分组逻辑
    same_mask = np.array([
        node_communities.get(r, -1) == node_communities.get(c, -2) 
        for r, c in zip(rows_all, cols_all)
    ])

    def find_active_cluster(log_v, raw_v, r_idx, c_idx):
        if len(log_v) == 0: return {}, {}
        
        # 在 Log 空间初始化中心点
        fixed_centroid = np.min(log_v) 
        centroid = np.max(log_v)
        
        is_stable = False
        while not is_stable:
            dist_to_fixed = np.abs(log_v - fixed_centroid)
            dist_to_active = np.abs(log_v - centroid)
            active_mask = dist_to_active < dist_to_fixed
            
            new_centroid = np.mean(log_v[active_mask]) if np.any(active_mask) else centroid
            if abs(new_centroid - centroid) < tolerance:
                is_stable = True
            centroid = new_centroid
            
        final_mask = (np.abs(log_v - centroid) < np.abs(log_v - fixed_centroid))
        
        # 构造两个簇的字典
        active_dict = dict(zip(zip(r_idx[final_mask], c_idx[final_mask]), raw_v[final_mask]))
        fixed_dict = dict(zip(zip(r_idx[~final_mask], c_idx[~final_mask]), raw_v[~final_mask]))
        return active_dict, fixed_dict

    # 3. 分类执行聚类
    cluster_same, fixed_same = find_active_cluster(
        all_log_values[same_mask], all_raw_values[same_mask], rows_all[same_mask], cols_all[same_mask]
    )
    cluster_diff, fixed_diff = find_active_cluster(
        all_log_values[~same_mask], all_raw_values[~same_mask], rows_all[~same_mask], cols_all[~same_mask]
    )

    # 4. 合并并返回两个字典 (解决解包报错)
    return {**cluster_same, **cluster_diff}, {**fixed_same, **fixed_diff}

def modified_kmeans_fast(mi_matrix, tolerance=1e-7):
    n = mi_matrix.shape[0]
    
    # 1. 提取上三角非零元素 (因为 mi_matrix 对称，且 j > i)
    # 这一步将 O(n^2) 的搜索范围缩小到实际有效的值
    triu_indices = np.triu_indices(n, k=1)
    all_values = mi_matrix[triu_indices]
    
    # 过滤掉 <= 0 的值 (对应原代码 if mi_matrix[i,j] <= 0: continue)
    valid_mask = all_values > 0
    values = all_values[valid_mask]
    # 记录这些有效值在原矩阵中的位置，最后还原字典用
    rows = triu_indices[0][valid_mask]
    cols = triu_indices[1][valid_mask]

    # 2. 初始化中心点
    fixed_centroid = 0.0
    centroid = np.max(values) if len(values) > 0 else 0.0
    
    is_stable = False
    
    while not is_stable:
        # 3. 向量化分类：计算每个点到两个中心点的距离
        # 原条件: (val - fixed_centroid) <= abs(val - centroid)
        dist_to_fixed = np.abs(values - fixed_centroid)
        dist_to_active = np.abs(values - centroid)
        
        # active_mask 为 True 表示该值应属于 cluster (动态簇)
        # 对应原代码 else 分支
        active_mask = dist_to_active < dist_to_fixed
        
        # 4. 更新动态中心点 (centroid)
        if np.any(active_mask):
            new_centroid = np.mean(values[active_mask])
        else:
            new_centroid = centroid
            
        # 5. 检查收敛条件：中心点偏移量小于阈值则稳定
        if abs(new_centroid - centroid) < tolerance:
            is_stable = True
        
        centroid = new_centroid

    # 6. 一次性还原为字典输出 (如果你的后续逻辑必须用字典)
    # 注意：如果 n 非常大，建议直接返回 mask 以节省内存
    fixed_mask = ~active_mask
    
    cluster = dict(zip(zip(rows[active_mask], cols[active_mask]), values[active_mask]))
    fixed_cluster = dict(zip(zip(rows[fixed_mask], cols[fixed_mask]), values[fixed_mask]))

    return cluster, fixed_cluster

def fast_mi_and_prob(x):
    # 假设 x 的形状是 (n_features, m_samples)
    n, m = x.shape
    
    # 1. 计算每个变量为 1 和 0 的概率
    # 使用 .reshape(-1) 确保它们是一维数组，方便后续计算
    count_1 = x.sum(axis=1).get() if hasattr(x, 'get') else x.sum(axis=1)
    count_1 = count_1.astype(float)
    count_0 = m - count_1
    
    p_i1 = count_1 / m
    p_i0 = count_0 / m

    # 2. 计算联合分布计数 (n x n)
    # count_11[i, j] 是 i=1 且 j=1 的样本数
    count_11 = x @ x.T
    
    # 3. 这里的 count_1 是一维的 (n,)，利用广播机制计算其他组合
    # count_1[:, None] 将其变为 (n, 1)
    count_1_col = count_1[:, np.newaxis]
    count_1_row = count_1[np.newaxis, :]
    
    count_10 = count_1_col - count_11
    count_01 = count_1_row - count_11
    count_00 = m - (count_11 + count_10 + count_01)

    # 4. 计算条件概率矩阵 p[i, j] = p(j=1 | i=1)
    # 注意：这里 i 是行，j 是列。原代码逻辑 p[i,j] = p_i1_j1 / p_i1
    p_matrix = count_11 / (count_1_col + 1e-12)

    # 5. 计算互信息 MI
    mi_matrix = np.zeros((n, n))
    
    # 组合列表：(联合概率, 边际概率1, 边际概率2)
    # p_i 和 p_j 均为形状为 (n,) 的一维数组
    pairs = [
        (count_11, p_i1, p_i1), # (1,1)
        (count_10, p_i1, p_i0), # (1,0)
        (count_01, p_i0, p_i1), # (0,1)
        (count_00, p_i0, p_i0)  # (0,0)
    ]

    for c_ij, p_i_vec, p_j_vec in pairs:
        p_ij = c_ij / m
        # 计算边际概率的乘积矩阵 P(i)*P(j)
        # np.outer(p_i_vec, p_j_vec) 会生成 (n, n) 矩阵
        p_i_p_j = np.outer(p_i_vec, p_j_vec)
        
        # 掩码计算：只有当联合概率和边际概率乘积均大于 0 时才计算
        mask = (p_ij > 1e-12) & (p_i_p_j > 1e-12)
        
        # MI 公式项
        mi_matrix[mask] += p_ij[mask] * np.log(p_ij[mask] / p_i_p_j[mask])

    return p_matrix, mi_matrix

def fast_imi_and_prob(x):
    # 假设 x 的形状是 (n_features, m_samples)
    if hasattr(x, 'get'): x = x.get() # 如果是 cupy 数组转为 numpy
    n, m = x.shape
    
    # 1. 计算每个变量为 1 和 0 的概率
    count_1 = x.sum(axis=1).astype(float)
    count_0 = m - count_1
    
    p_i1 = count_1 / m
    p_i0 = count_0 / m

    # 2. 计算联合分布计数 (n x n)
    count_11 = x @ x.T
    
    # 3. 利用广播机制计算其他组合
    count_1_col = count_1[:, np.newaxis]
    count_1_row = count_1[np.newaxis, :]
    
    count_10 = count_1_col - count_11
    count_01 = count_1_row - count_11
    count_00 = m - (count_11 + count_10 + count_01)

    # 4. 计算条件概率矩阵 p[i, j] = p(j=1 | i=1)
    p_matrix = count_11 / (count_1_col + 1e-12)

    # 5. 计算 IMI
    imi_matrix = np.zeros((n, n))
    
    # 定义四个分量的配置：(联合计数, 行边缘概率, 列边缘概率, 是否为负贡献)
    # 这里的贡献符号 sign 对应公式中的 + 或 -
    components = [
        (count_11, p_i1, p_i1, 1),  # MI(1,1) -> 正向
        (count_00, p_i0, p_i0, 1),  # MI(0,0) -> 正向
        (count_10, p_i1, p_i0, -1), # -|MI(1,0)| -> 负向
        (count_01, p_i0, p_i1, -1)  # -|MI(0,1)| -> 负向
    ]

    for c_ij, p_row_vec, p_col_vec, sign in components:
        p_ij = c_ij / m
        # 计算 P(Xi)*P(Xj) 矩阵
        p_i_p_j = np.outer(p_row_vec, p_col_vec)
        
        # 避免 log(0) 或 除以 0
        mask = (p_ij > 1e-12) & (p_i_p_j > 1e-12)
        
        # 计算单项 MI
        term = np.zeros((n, n))
        term[mask] = p_ij[mask] * np.log(p_ij[mask] / p_i_p_j[mask])
        
        # 根据 sign 累加到最终矩阵
        if sign == 1:
            imi_matrix += term
        else:
            # 公式要求减去绝对值: -|MI|
            imi_matrix -= np.abs(term)

    return p_matrix, imi_matrix

def IC(Networkx_Graph, Seed_Set, Probability):

    tree = nx.DiGraph()
    tree.add_node(Seed_Set[0])
    new_active, Ans = Seed_Set.tolist(), Seed_Set.tolist()
    while new_active:
        # Getting neighbour nodes of newly activate node
        (targets, edges) = Neighbour_finder(Networkx_Graph, Probability, new_active)
        # Calculating if any nodes of those neighbours can be activated, if yes add them to new_ones.

        new_active = []

        for (node, target) in edges:
            if np.random.uniform(0, 1) < Probability[node, target]:
                if target not in Ans: #success infected
                    tree.add_edge(node, target)
                    new_active.append(target)
                    Ans.append(target)
        # Checking which ones in new_ones are not in our Ans...only adding them to our Ans so that no duplicate in Ans.

    return Ans, tree


def Neighbour_finder(g, p, new_active):
    targets = []
    edges = []
    for node in new_active:
        node_neighbors = list(g.neighbors(node))
        targets += node_neighbors
        for target in node_neighbors:
            edges.append((node,target))

    return (targets, edges)

def generate_infections(A, num_sim = 100):

    N = A.shape[0]
    S = np.zeros([num_sim, N])
    nx_graph = nx.from_numpy_array(A)
    trees = []
    while len(trees) < num_sim:
        seed = np.random.choice(np.arange(0, N), size=1)
        cascade, tree = IC(Networkx_Graph=nx_graph, Seed_Set=seed, Probability=A)
        if len(tree.nodes) >= 3:
            S[len(trees), cascade] = 1
            trees.append(tree)
    average_paths = 0
    for tree in trees:
        average_paths += len(tree.nodes())

    print("average length of infections: ", average_paths / len(trees))
    return S

In [ ]:
edges = set()
data_path = '../dataset/email-Eu-core/'
with open(data_path+'email-Eu-core.txt', 'r') as f:
    for l in f:
        if l.strip() != '':
            edges.add((int(l.strip().split(' ')[0]), int(l.strip().split(' ')[1])))
G = nx.DiGraph()
G.add_edges_from(edges)
N = len(G)
node_communities = {}

with open(data_path + 'email-Eu-core-department-labels.txt', 'r') as f:
    for l in f:
        if l.strip() != '':
            node_communities[int(l.strip().split(' ')[0])] = int(l.strip().split(' ')[1])

A = nx.to_numpy_array(G)
P = np.zeros((N, N))
for u, v in G.edges():
    if node_communities[u] == node_communities[v]:
        weight = np.random.uniform(0.05, 0.1)
    else:
        weight = np.random.uniform(0.01, 0.05)
    P[u, v] = weight
    P[v, u] = weight
A = A * P
S = generate_infections(A, num_sim=1000)

average length of infections:  3.966


In [125]:
mi_matrix, p_matrix = fast_imi_and_prob(S.T)
cluster, fixed_cluster = modified_kmeans_fast_log_partitioned(mi_matrix, node_communities) #log 296
threshold = max(fixed_cluster.values())
prune_network = np.zeros([N, N])
prune_network[mi_matrix > threshold] = 1.0
prune_network[mi_matrix <= threshold] = 0.0

In [3]:
prune_network = np.ones([N, N])

In [126]:
G = nx.from_numpy_array(A)

In [127]:
#check edge

def check_pruned_edges(G, prune_network):
    """
    检查图 G 中的边有多少被 prune_network 过滤掉了
    """
    # 1. 获取 G 中所有的边
    original_edges = list(G.edges())
    total_g_edges = len(original_edges)
    
    missing_edges = []
    
    # 2. 遍历 G 的边，检查在矩阵中的对应位置是否为 0
    for u, v in original_edges:
        # 确保索引不越界
        if u < prune_network.shape[0] and v < prune_network.shape[1]:
            if prune_network[u, v] == 0:
                missing_edges.append((u, v))
        else:
            # 如果节点索引超出了矩阵范围，记录为异常
            print(f"Warning: Node index ({u}, {v}) out of prune_network bounds.")

    # 3. 计算统计数据
    num_missing = len(missing_edges)
    missing_ratio = (num_missing / total_g_edges) * 100 if total_g_edges > 0 else 0
    
    print("-" * 30)
    print(f"原始图 G 总边数: {total_g_edges}")
    print(f"被剪枝掉的边数 (不在 prune_network 中): {num_missing}")
    print(f"漏掉比例 (FN 潜在来源): {missing_ratio:.2f}%")
    print("-" * 30)
    
    return missing_edges

missing = check_pruned_edges(G, prune_network)

------------------------------
原始图 G 总边数: 1636
被剪枝掉的边数 (不在 prune_network 中): 429
漏掉比例 (FN 潜在来源): 26.22%
------------------------------


In [128]:
count = np.sum(prune_network == 1)
print(count)

8216


In [144]:
import torch
import torch.nn as nn
import torch.optim as optim
import tqdm
from torch.linalg import inv, slogdet
import numpy as np
import networkx as nx
from kneed import KneeLocator
import torch.nn.functional as F

class RegularizedInferenceIC(nn.Module):
    def __init__(self, N, Cascades, InstancePartition, gamma, prune_network):
        super(RegularizedInferenceIC, self).__init__()
        
        self.N = N
        self.gamma = gamma
        
        # 优化参数：网络边的概率对数 (对应 alpha)
        self.A_param = nn.Parameter(torch.zeros((N, N)))
        
        # 处理剪枝网络并注册为 buffer
        prune_network[prune_network == 0] = 1e-5
        self.register_buffer('prune_network_tensor', torch.from_numpy(prune_network).float())

        # =================================================================
        # 预计算 1: 级联状态矩阵 X (直接利用 GPU 矩阵乘法替代循环)
        # =================================================================
        # Cascades 形状为 (L, N)，即 L 个级联，N 个节点。X_li 表示级联 l 中节点 i 的状态
        X_np = np.array(Cascades)
        self.register_buffer('X', torch.tensor(X_np, dtype=torch.float32))
        
        #+++2
        co_occurrence = np.dot(X_np.T, X_np)
        # 归一化，避免数值过大
        co_occurrence = co_occurrence / (X_np.shape[0] + 1e-8)
        # 将共现频率映射到参数空间
        # 因为后面用 softplus，所以这里可以做个逆映射或简单的线性缩放
        # 我们希望共现高的边，初始 w 更大
        init_weight = torch.from_numpy(co_occurrence).float() * 0.5
        self.A_param = nn.Parameter(init_weight)

        # =================================================================
        # 预计算 2: 实例划分掩码矩阵 M (用于计算正则化 Omega)
        # =================================================================
        unique_insts = list(set(InstancePartition.values()) if isinstance(InstancePartition, dict) else set(InstancePartition))
        num_insts = len(unique_insts)
        M = torch.zeros(N, num_insts, dtype=torch.float32)
        
        for u in range(N):
            inst_id = InstancePartition[u] if isinstance(InstancePartition, dict) else InstancePartition[u]
            idx = unique_insts.index(inst_id)
            M[u, idx] = 1.0
            
        self.register_buffer('M_matrix', M)
        
        counts = M.sum(dim=0).unsqueeze(1)
        counts[counts == 0] = 1.0 
        self.register_buffer('M_counts', counts)

    def _get_prob_matrix(self):
        A_prob = torch.sigmoid(self.A_param)
        # 屏蔽对角线并应用剪枝网络
        A_prob = A_prob * (1.0 - torch.eye(self.N, device=A_prob.device))
        A_prob = A_prob * self.prune_network_tensor
        return A_prob
    
    #+++2
    def _get_weights(self):
        # =================================================================
        # 优化 2: 直接建模 w (Softplus)
        # softplus(x) = log(1 + exp(x))，保证 w > 0 且在大值区梯度不消失
        # =================================================================
        W = F.softplus(self.A_param)
        
        # 应用剪枝和对角线屏蔽
        W = W * (1.0 - torch.eye(self.N, device=W.device))
        W = W * self.prune_network_tensor
        return W

    def forward(self):
        #+++2
        # A_prob = self._get_prob_matrix()
        eps = 1e-8
        
        # =================================================================
        # 1. 负对数似然 (NLL) 高速张量计算 (对应公式推导)
        # =================================================================
        # w_ij = -log(1 - alpha_ij)
        #+++2
        W = self._get_weights()
        A_prob = -torch.expm1(-W)
        
        # 计算 y_i^l = \sum_j x_j^l w_ij
        # X 形状 (L, N)，W.T 形状 (N, N)，结果 Y 形状 (L, N)
        Y = torch.matmul(self.X, W.T)
        
        # 计算 Loss = sum [ (1 - X) * Y  -  X * log(1 - e^{-Y}) ]
        # 项 1: -(1 - x_i^l)y_i^l 的相反数
        term1 = (1.0 - self.X) * Y
        
        #+++3
        denominator = Y + eps
        log_prob_active = torch.log(-torch.expm1(-Y) + eps)
        term2 = self.X * log_prob_active
        # 项 2: x_i^l * log(1 - e^{-y_i^l}) 的相反数
        # term2 = self.X * torch.log(1.0 - torch.exp(-Y) + eps)
        
        #+++1
        negative_mask = ((1.0 - self.X) * Y > 0).float()
        # 对负样本进行随机下采样 (假设只保留 10% 的负样本惩罚)
        sampling_rate = 0.1
        random_mask = (torch.rand_like(Y) < sampling_rate).float()
        effective_negative_mask = negative_mask * random_mask
        term1_sampled = term1 * effective_negative_mask
        
        # 对所有级联和节点求和，并除以总数做归一化，防止 loss 爆掉
        NLL = torch.sum(term1_sampled - term2) / (self.N * self.N)
        
        # =================================================================
        # 2. 高速计算 Regularization (Omega)
        # =================================================================
        A_sum = torch.matmul(self.M_matrix.T, A_prob)
        A_mean = A_sum / self.M_counts 
        A_approx = torch.matmul(self.M_matrix, A_mean) 
        Omega = torch.sum((A_prob - A_approx)**2) / (self.N * self.N) 
        
        # 3. 返回最终的损失
        return NLL + self.gamma * Omega


def post_processing(estimated_A, beta=1.0):
    """
    beta: Recall 偏置系数。
    beta > 1.0 会让模型更厌恶漏报(FN)，从而降低阈值提高 Recall。
    """
    thresholds = np.linspace(start=1e-6, stop=0.5, num=10000)
    FP_FN_diff = np.zeros([len(thresholds)])

    for i, t in enumerate(thresholds):
        # 估计的漏报项 (本应是边但被阈值切掉了)
        predicted_FN = np.sum(estimated_A[estimated_A < t])
        # 估计的误报项 (本不该是边但被保留了)
        predicted_FP = np.sum(1.0 - estimated_A[estimated_A >= t])

        # 修改点：通过权重 beta 强迫模型降低阈值
        # 当 beta=2.0 时，1个 FN 的代价等于 2个 FP
        FP_FN_diff[i] = np.abs(predicted_FP - beta * predicted_FN)

    best_t = thresholds[np.argmin(FP_FN_diff)]
    
    IG_mat = np.zeros_like(estimated_A)
    IG_mat[estimated_A >= best_t] = 1
    IG = nx.from_numpy_array(IG_mat)

    return best_t, IG

def post_processing_kneed(estimated_A):
    # 1. 获取排序后的概率
    probs = np.sort(estimated_A.flatten())[::-1]
    probs = probs[probs > 1e-5]
    x = np.arange(len(probs))
    
    # 2. 调用 KneeLocator
    # curve='convex': 曲线是凸的
    # direction='decreasing': 曲线是递减的
    kneedle = KneeLocator(x, probs, S=1.0, curve='convex', direction='decreasing')
    
    # 3. 获取拐点对应的索引和阈值
    best_idx = kneedle.knee # 拐点的索引
    if best_idx is None:
        best_t = 0.05 # 备选保守阈值
    else:
        best_t = probs[best_idx]
        
    # 4. 绘图展示 (可选，调试时非常有用)
    # kneedle.plot_knee() 
    
    IG_mat = (estimated_A >= best_t).astype(float)
    return best_t, nx.from_numpy_array(IG_mat)

def calculate_F1(IG,RG):

    ig_edges = IG.edges
    rg_edges = RG.edges

    TP = 0.0
    FP = 0.0
    FN = 0.0

    for (i,j) in ig_edges:
        if (i,j) in rg_edges or (j,i) in rg_edges:
            TP += 1.0
        else:
            FP += 1.0

    for (i,j) in rg_edges:
        if (i,j) not in ig_edges and (j,i) not in ig_edges:
            FN += 1.0
    
    print(TP,FP,FN)

    P = TP / (TP+FP)
    R = TP / (TP+FN)

    return round(P,3),round(R,3),round(2*P*R / (P+R),3)

In [88]:
C = node_communities

    
l = set()
for node in C:
    l.add(C[node])
print(len(l))

dict_c = dict()
for i, item in enumerate(l):
    dict_c[item] = i
    
for node in C:
    C[node] = dict_c[C[node]]
    
gamma = 0.05

42


In [146]:
iterations=2000
lr=0.005
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")
G = nx.from_numpy_array(A)
np.fill_diagonal(prune_network, 0.0)

# 模型初始化时所有的预计算矩阵会被创建
model = RegularizedInferenceIC(N=N, 
                                Cascades=S, 
                                InstancePartition=C, 
                                gamma=gamma,
                                prune_network=prune_network).to(device)

optimizer = optim.Adam(model.parameters(), lr=lr) 

model.train()
for i in tqdm.tqdm(range(iterations), desc="Optimizing"):
    optimizer.zero_grad()
    loss = model() 
    loss.backward() 
    optimizer.step() 
    
    # 为了避免影响 tqdm 的输出，建议降低打印频率 (比如每50轮打印一次)
    if (i+1) % 50 == 0:
        tqdm.tqdm.write(f"Iteration {i+1}, Loss: {loss.item():.4f}")

model.eval()
with torch.no_grad():
    A_star = model._get_prob_matrix().cpu().numpy()
    
A_star = A_star * prune_network
A_star[A_star <= 1e-5] = 0.0

# np.savetxt('A_star_matrix.csv', A_star, delimiter=',')
# np.savetxt('prune_network.csv', prune_network, delimiter=',')
best_t, IG = post_processing(A_star)
P, R, F1 = calculate_F1(IG, G) # 未剪枝TP,FP,FN 8587.0 454416.0 12605.0 剪枝 95.0 62.0 25421.0 社区不剪枝 1984.0 10015.0 22515.0 //0.401
print(f"BEP point : {best_t:.5f} | P: {P}, R: {R}, F1: {F1}")

Training on device: cpu


Optimizing:   0%|          | 3/2000 [00:00<01:25, 23.42it/s]

Optimizing:   3%|▎         | 54/2000 [00:02<01:23, 23.26it/s]

Iteration 50, Loss: 0.0104


Optimizing:   5%|▌         | 102/2000 [00:04<01:28, 21.45it/s]

...

Iteration 1900, Loss: 0.0032


Optimizing:  98%|█████████▊| 1954/2000 [01:18<00:01, 26.77it/s]

Iteration 1950, Loss: 0.0033


Optimizing: 100%|██████████| 2000/2000 [01:19<00:00, 25.09it/s]


Iteration 2000, Loss: 0.0032
575.0 725.0 1061.0
BEP point : 0.42019 | P: 0.442, R: 0.351, F1: 0.392
